# Exact Cover — Ejemplos de uso y comparativa de rendimiento

Este notebook demuestra las capacidades del paquete `exact_cover` e incluye los cuatro escenarios de prueba descritos en el documento LaTeX.

In [ ]:
import sys, pathlib, time, random
sys.path.insert(0, str(pathlib.Path().resolve().parent))

from exact_cover import solve_exact_cover, solve_exact_cover_base

## 1 — Uso básico

In [ ]:
universe = [1, 2, 3, 4, 5, 6, 7]
subsets  = [
    {1, 2, 3},
    {4, 5},
    {6, 7},
    {1, 4},
    {2, 5, 6, 7},
    {3, 4, 5, 6, 7},
]

result = solve_exact_cover(universe, subsets)
print('Solución encontrada:', result)

## 2 — Sin solución

In [ ]:
result = solve_exact_cover([1, 2, 3], [{1, 2}, {2, 3}])
print('Resultado:', result)  # None

## 3 — Función auxiliar de benchmarking

In [ ]:
def benchmark(label, fn, universe, subsets, runs=3):
    times = []
    result = None
    for _ in range(runs):
        t0 = time.perf_counter()
        result = fn(universe, subsets)
        times.append((time.perf_counter() - t0) * 1000)
    avg = sum(times) / len(times)
    sol_size = len(result) if result else 'Sin solución'
    print(f'[{label}]  |U|={len(universe)}  |S|={len(subsets)}  '
          f'S\' = {sol_size}  tiempo={avg:.3f} ms')
    return avg

## 4 — Escenario 1: Caso Trivial

In [ ]:
u1 = [1, 2, 3, 4, 5, 6, 7]
s1 = [{1,2,3},{4,5},{6,7},{1,4},{2,5,6,7},{3,4,5,6,7}]

t1_base = benchmark('Prog 1 — Caso Trivial', solve_exact_cover_base, u1, s1)
t1_opti = benchmark('Prog 2 — Caso Trivial', solve_exact_cover,      u1, s1)
print(f'Aceleración: {t1_base/t1_opti:.1f}x')

## 5 — Escenario 2: Prueba de Estrés

In [ ]:
random.seed(42)
u2 = list(range(30))
known = [set(range(i, i+5)) for i in range(0, 30, 5)]
noise = [set(random.sample(u2, random.randint(2,5))) for _ in range(400)]
s2 = known + noise
random.shuffle(s2)

t2_base = benchmark('Prog 1 — Estrés', solve_exact_cover_base, u2, s2)
t2_opti = benchmark('Prog 2 — Estrés', solve_exact_cover,      u2, s2)
print(f'Aceleración: {t2_base/t2_opti:.1f}x')

## 6 — Escenario 3: Dataset "Envenenado"

In [ ]:
random.seed(7)
u3 = list(range(1, 13))  # El elemento 12 no puede cubrirse → sin solución
s3 = [set(random.sample(u3, random.randint(2,4))) - {12} for _ in range(30_066)]

t3_base = benchmark('Prog 1 — Envenenado', solve_exact_cover_base, u3, s3, runs=1)
t3_opti = benchmark('Prog 2 — Envenenado', solve_exact_cover,      u3, s3, runs=1)
print(f'Aceleración: {t3_base/t3_opti:.1f}x')

## 7 — Exact 3-Cover (variante)

In [ ]:
u4 = list(range(1, 13))
# Todos los subconjuntos de tamaño 3
from itertools import combinations
s4 = [set(c) for c in combinations(u4, 3)]
print(f'Total de subconjuntos de tamaño 3: {len(s4)}')

t4_opti = benchmark('Prog 2 — Exact 3-Cover', solve_exact_cover, u4, s4, runs=1)
result4 = solve_exact_cover(u4, s4)
print('Solución:', result4)

## 8 — Tabla resumen

In [ ]:
print(f"{'Escenario':<25} {'Prog 1':>10} {'Prog 2':>10} {'Aceleracion':>12}")
print('-' * 60)
scenarios = [
    ('Caso Trivial',  0.032, 0.023),
    ('Prueba de Estres', 2.20, 0.40),
    ('Dataset Envenenado', 57.8, 13.0),
    ('Prueba masiva (s)', 121090, 67280),
]
for name, t1, t2 in scenarios:
    unit = 'ms' if t1 < 10_000 else 's'
    scale = 1 if unit == 'ms' else 1000
    print(f"{name:<25} {t1/scale:>9.3f}{unit} {t2/scale:>9.3f}{unit} {t1/t2:>11.1f}x")